# ⚡ GAN PLAYGROUND: SÂN CHƠI AI TẠO SINH DÀNH CHO HỌC SINH
### Khám phá Top 3 Ứng dụng Kỳ diệu của GAN: CycleGAN, Pix2Pix & StyleGAN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDanh1510/GAN-playground/blob/main/notebooks/top3_gan_playground.ipynb)

---

## 🎯 Mục Tiêu Khóa Học Thực Hành:
1. Hiểu bản chất trò chơi **Kẻ Làm Giả (Generator)** vs **Cảnh Sát (Discriminator)** trong GAN.
2. Tự tay chạy **Top 3 Ứng dụng AI Tạo sinh** nổi tiếng nhất thế giới:
   - 🦓 **CycleGAN**: Biến đổi thế giới không cần ghép đôi dữ liệu (*Ngựa $\leftrightarrow$ Ngựa vằn, Mùa hè $\leftrightarrow$ Mùa đông*).
   - 🎨 **Pix2Pix**: Bút vẽ ma thuật biến nét phác thảo doodle thành tranh vẽ hoàn chỉnh.
   - 😎 **StyleGAN Latent Studio**: Đại số vector mặt người (*Mặt thường + Vector cười = Mặt cười*).
3. Nắm vững **10 Mẹo thực chiến (GAN Hacks)** của Soumith Chintala để huấn luyện GAN luôn ổn định!

### 0. Kiểm tra phần cứng & Thư viện (GPU Acceleration)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Đang sử dụng thiết bị: {device}")
if torch.cuda.is_available():
    print(f"🔥 Tên GPU: {torch.cuda.get_device_name(0)}")
else:
    print("ℹ️ Đang chạy trên CPU (vẫn chạy mượt mà do code đã được tối ưu nhẹ nhàng!)")

---
## 💖 Phần 1: 2D Point GAN – Cơ Chế Đối Kháng Lõi Trong 35 Dòng Code

> **Ý tưởng**: Generator nhận một điểm nhiễu ngẫu nhiên $z \sim \mathcal{N}(0, 1)$ và cố gắng biến đổi nó thành một điểm nằm trên **Đường cong Trái tim** để lừa Discriminator.

In [ ]:
# 1. Định nghĩa Generator & Discriminator
class Generator2D(nn.Module):
    def __init__(self, latent_dim=2, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 2)
        )
    def forward(self, z):
        return self.net(z)

class Discriminator2D(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

# 2. Sinh tập dữ liệu hình trái tim
def get_heart_distribution(batch_size=256):
    t = torch.rand(batch_size) * 2 * np.pi - np.pi
    x = 16 * (torch.sin(t) ** 3) / 18
    y = (13 * torch.cos(t) - 5 * torch.cos(2*t) - 2 * torch.cos(3*t) - torch.cos(4*t)) / 18
    noise = torch.randn(batch_size, 2) * 0.04
    return torch.stack([x, y], dim=1) + noise

# 3. Huấn luyện 2D GAN
G2D = Generator2D().to(device)
D2D = Discriminator2D().to(device)
criterion = nn.BCELoss()
opt_G = optim.Adam(G2D.parameters(), lr=0.006, betas=(0.5, 0.999))
opt_D = optim.Adam(D2D.parameters(), lr=0.006, betas=(0.5, 0.999))

print("⚡ Đang huấn luyện 2D GAN...")
for epoch in range(1, 201):
    # Train Discriminator
    real_pts = get_heart_distribution(256).to(device)
    noise = torch.randn(256, 2, device=device)
    fake_pts = G2D(noise)
    
    d_loss = (criterion(D2D(real_pts), torch.ones(256, 1, device=device)) +
              criterion(D2D(fake_pts.detach()), torch.zeros(256, 1, device=device))) / 2
    opt_D.zero_grad(); d_loss.backward(); opt_D.step()
    
    # Train Generator
    g_loss = criterion(D2D(fake_pts), torch.ones(256, 1, device=device))
    opt_G.zero_grad(); g_loss.backward(); opt_G.step()

# 4. Trực quan hóa kết quả
with torch.no_grad():
    real_demo = get_heart_distribution(400).cpu().numpy()
    fake_demo = G2D(torch.randn(400, 2, device=device)).cpu().numpy()

plt.figure(figsize=(6, 6))
plt.scatter(real_demo[:, 0], real_demo[:, 1], c='#f97316', s=14, label='Dữ liệu Thật (Trái tim)', alpha=0.7)
plt.scatter(fake_demo[:, 0], fake_demo[:, 1], c='#06b6d4', s=14, label='Dữ liệu Máy vẽ (Generator)', alpha=0.7)
plt.title('Kết Quả 2D GAN Học Hình Trái Tim')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 🎨 Phần 2: Pix2Pix – Bút Vẽ Phù Thủy (Sketch-to-Image)

> **Nguyên lý U-Net & PatchGAN**:
- **U-Net Generator**: Có các đường nối tắt (*Skip Connections*) giúp bảo toàn các chi tiết nét vẽ sắc cạnh từ lớp đầu đến lớp cuối.
- **PatchGAN Discriminator**: Không chấm điểm toàn bộ bức ảnh một lúc mà chia nhỏ ảnh thành các ô vuông $N \times N$ (patches) để soi xét chi tiết từng vùng.

In [ ]:
# 1. U-Net Generator Mini
class MiniUNet(nn.Module):
    def __init__(self, in_c=1, out_c=3):
        super().__init__()
        self.e1 = nn.Conv2d(in_c, 32, 4, 2, 1) # 64 -> 32
        self.e2 = nn.Sequential(nn.Conv2d(32, 64, 4, 2, 1, bias=False), nn.BatchNorm2d(64), nn.LeakyReLU(0.2))
        self.e3 = nn.Sequential(nn.Conv2d(64, 128, 4, 2, 1, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2))
        
        self.d1 = nn.Sequential(nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU())
        self.d2 = nn.Sequential(nn.ConvTranspose2d(64 + 64, 32, 4, 2, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU())
        self.d3 = nn.Sequential(nn.ConvTranspose2d(32 + 32, out_c, 4, 2, 1), nn.Tanh())
        
    def forward(self, x):
        e1 = nn.LeakyReLU(0.2)(self.e1(x))
        e2 = self.e2(e1)
        e3 = self.e3(e2)
        d1 = self.d1(e3)
        d2 = self.d2(torch.cat([d1, e2], dim=1)) # Skip Connection
        return self.d3(torch.cat([d2, e1], dim=1))

# 2. Sinh tập dữ liệu phác thảo
def get_sketch_batch(batch_size=8, img_size=64):
    sketches = torch.zeros(batch_size, 1, img_size, img_size)
    colors = torch.zeros(batch_size, 3, img_size, img_size)
    for i in range(batch_size):
        cx, cy = np.random.randint(20, 44, 2)
        r = np.random.randint(12, 18)
        color = np.random.rand(3) * 2 - 1
        y, x = np.ogrid[:img_size, :img_size]
        dist = np.sqrt((x - cx)**2 + (y - cy)**2)
        sketches[i, 0, (dist >= r-2) & (dist <= r+2)] = 1.0
        for c in range(3): colors[i, c, dist <= r] = color[c]
    return sketches, colors

pix_G = MiniUNet().to(device)
print("✓ Khởi tạo Pix2Pix U-Net Generator thành công!")

# Chạy thử nghiệm suy luận 1 batch nét vẽ
demo_sketches, demo_reals = get_sketch_batch(4)
with torch.no_grad():
    demo_fakes = pix_G(demo_sketches.to(device)).cpu()

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i in range(4):
    axes[0, i].imshow(demo_sketches[i, 0], cmap='gray')
    axes[0, i].set_title(f"Nét phác thảo {i+1}"); axes[0, i].axis('off')
    axes[1, i].imshow(((demo_reals[i].permute(1, 2, 0) + 1) / 2).clip(0, 1))
    axes[1, i].set_title(f"Ảnh màu mục tiêu"); axes[1, i].axis('off')
plt.suptitle("Dữ liệu Huấn Luyện Pix2Pix (Cặp Nét vẽ ↔ Ảnh màu)")
plt.show()

---
## 🦓 Phần 3: CycleGAN – Máy Biến Đổi Thế Giới (Unpaired Translation)

> **Vòng lặp dịch thuật khép kín (Cycle Consistency)**:
$$\mathcal{L}_{cycle} = \|F(G(A)) - A\|_1 + \|G(F(B)) - B\|_1$$
- Generator $G: A \to B$ (Biến Ngựa thành Ngựa vằn)
- Generator $F: B \to A$ (Biến Ngựa vằn ngược lại thành Ngựa thường)
- Ảnh gốc $A \xrightarrow{G} B' \xrightarrow{F} A''$ phải giống hệt $A$ ban đầu!

In [ ]:
# Kiểm tra công thức Cycle Consistency Loss bằng mã PyTorch
real_A = torch.randn(2, 3, 32, 32) # Ảnh miền A (Ngựa)
fake_B = torch.randn(2, 3, 32, 32) # Ảnh sau khi biến đổi sang B (Ngựa vằn)
rec_A = real_A + torch.randn_like(real_A) * 0.05 # Tái tạo lại sau chu trình khép kín

l1_loss = nn.L1Loss()
cycle_loss_val = l1_loss(rec_A, real_A)
print(f"✨ Điểm lệch Cycle Consistency Loss: {cycle_loss_val.item():.4f}")
print("💡 Cycle Loss càng nhỏ, mô hình càng bảo toàn tốt hình dáng của đối tượng gốc!")

---
## 😎 Phần 4: StyleGAN Latent Studio – Đại Số Vector Mặt Người & Biến Hình (Morphing)

> **Đại số không gian tiềm ẩn (Latent Space Arithmetic)**:
$$\vec{z}_{kết\_quả} = \vec{z}_{gốc} + \alpha \cdot \vec{v}_{nụ\_cười} + \beta \cdot \vec{v}_{kính\_râm}$$

In [ ]:
# DCGAN Generator mini tạo khuôn mặt / emoji
class FaceGenerator(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128 * 4 * 4)
        self.conv = nn.Sequential(
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False), # 4x4 -> 8x8
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False), # 8x8 -> 16x16
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32, 3, 4, 2, 1),              # 16x16 -> 32x32
            nn.Tanh()
        )
    def forward(self, z):
        x = self.fc(z).view(-1, 128, 4, 4)
        return self.conv(x)

latent_dim = 32
G_face = FaceGenerator(latent_dim=latent_dim).to(device)
G_face.eval()

# 1. Đại số Vector
z_base = torch.randn(1, latent_dim, device=device)
v_smile = torch.randn(1, latent_dim, device=device) * 0.4
v_glasses = torch.randn(1, latent_dim, device=device) * 0.5

z_smile = z_base + 1.2 * v_smile
z_glasses = z_base + 1.5 * v_glasses
z_combo = z_base + 1.2 * v_smile + 1.5 * v_glasses

# 2. Morphing giữa Người A và Người B
z_A = torch.randn(1, latent_dim, device=device)
z_B = torch.randn(1, latent_dim, device=device)

alphas = np.linspace(0, 1, 6)
morph_imgs = []
with torch.no_grad():
    for a in alphas:
        z_interp = (1 - a) * z_A + a * z_B
        img = (G_face(z_interp)[0].permute(1, 2, 0).cpu() + 1) / 2
        morph_imgs.append(img.clamp(0, 1).numpy())

fig, axes = plt.subplots(1, 6, figsize=(14, 3))
for i, img in enumerate(morph_imgs):
    axes[i].imshow(img)
    axes[i].set_title(f"{int(alphas[i]*100)}% (A ➔ B)")
    axes[i].axis('off')
plt.suptitle("Quá Trình Biến Hình Mượt Mà Giữa 2 Người Trong Không Gian Tiềm Ẩn (Latent Morphing)")
plt.show()

---
## 🏆 10 Mẹo Huấn Luyện GAN Thành Công (GAN Hacks for Students)

1. **Chuẩn hóa ảnh về $[-1, 1]$**: Luôn dùng hàm kích hoạt `Tanh()` ở lớp cuối của Generator.
2. **Dùng LeakyReLU**: Thay vì `ReLU` thông thường, hãy dùng `LeakyReLU(0.2)` để gradient không bị chết.
3. **Lấy mẫu nhiễu từ phân phối Gauss**: Tạo $z$ bằng `torch.randn()` thay vì phân phối đều `torch.rand()`.
4. **Kỹ thuật Label Smoothing**: Đặt nhãn ảnh thật là `0.9` thay vì `1.0` để tránh Discriminator quá tự tin.
5. **Tránh Mode Collapse**: Theo dõi biểu đồ Loss; nếu $G$ Loss tăng vọt và ảnh sinh ra giống hệt nhau, hãy giảm learning rate của $D$ hoặc áp dụng Wasserstein GAN (WGAN)!